|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 6:</h2>|<h1>The Server<h1>|
|<h2>Section:</h2>|<h1>Incidents<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge: the incident file<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

You finished Part 6. The engine is a server, and the server has metrics. Now
the failures come from the layers around the engine: the event loop, the
network, the proxy, and the dashboard. Several tickets in this file are about
a number that is wrong, not about an engine that is wrong.

Each ticket gives you a **symptom** and some **evidence**. Some of the
evidence is noise. Write four lines for each ticket:

1. **Root cause.** One sentence.
2. **The number that proves it.** Not "it looks like". A computation.
3. **The fix.**
4. **The guard.** A test, an assert or an alert that catches it next time.

Four rules:

- The tickets are **not** in the order of the notebooks.
- At least one ticket is **not a bug**. "Nothing is broken" is a valid answer
  only if a number proves it.
- Write your answer **before** you open the solution.
- Every ticket has a scratch cell.

Do this section after stage 16. This notebook needs no GPU.

**The on-call colleague.** In Claude Code, type `/incident 6.1` (or any
other ticket number) to work a ticket as a conversation. The colleague has
access to the system. Ask for a log, a measurement or an experiment, and it
answers with what the system shows. When you write your four lines, it tells
you which lines are weak, and it asks a question about each one. It does not
tell you the cause until you ask for the solution.

### The reference sheet

| Metric | Definition |
|---|---|
| TTFT | from the arrival of the request to its first token |
| TPOT | the average time between the output tokens of one request |
| ITL | each single time between two output tokens |
| goodput | the requests each second that meet their promise |

- Little's law: requests in the system = arrival rate x time in the system.
- A percentile of a union is not the average of the percentiles of the
  parts.
- Server-Sent Events (SSE) send each token as one small chunk of an HTTP
  response that stays open.

# Ticket 1: every stream freezes at the same moment

**Severity:** medium. **Reported by:** users.

> Several times each hour, all the streams stop for about half a second,
> and then continue.

**Evidence**

- The freezes appear in the client logs as a gap of 400 to 450 ms between
  two tokens, at the same wall-clock moment for every stream.
- The engine log shows a step time of 22 ms, flat, including during the
  freezes.
- The request handler:

  ```python
  @app.post('/v1/completions')
  async def completions(req: CompletionRequest):
      ids = tokenizer.encode(req.prompt)
      return StreamingResponse(engine.generate(ids, req.params))
  ```

- The freezes start when a user sends a document. Chunked prefill is on.

### Solution

- **Root cause.** `tokenizer.encode` is a slow, synchronous call inside an
  `async` handler. While it runs, the event loop can do nothing else: no
  token goes out, and the engine task cannot start the next step.
- **The number.** The freeze is 400 to 450 ms, and the tokenization of a
  document of 100,000 tokens takes 420 ms. The engine steps stay at 22 ms,
  so the delay is not on the GPU. It is between the steps.
- **The fix.** Run the tokenization off the event loop:
  `await loop.run_in_executor(pool, tokenizer.encode, prompt)`. Real vLLM
  goes further, and runs the API server and the engine in separate
  processes.
- **The guard.** A monitor of the event loop lag: schedule a task every
  10 ms, and alert when it runs late by more than 50 ms.

**The noise.** Chunked prefill. It protects the GPU step from a long
prefill, and the GPU step was never the problem.

**What you could ask the system**

The on-call colleague answers these questions. Each answer is a piece of evidence that the ticket does not show.

- *How long does `tokenizer.encode` take for a document of 100,000 tokens?*
  About 420 ms.
- *Does the engine run in the same process and thread as the handler?*
  The engine loop runs as a task in the same event loop.
- *Does the GPU go idle during a freeze?*
  Yes, for about 400 ms. The engine does not start the next step.

# Ticket 2: the KV cache is full at night

**Severity:** medium. **Reported by:** the capacity team.

> The KV cache is 95% full even at night, when few users are active. We
> need more GPU memory.

**Evidence**

- 30% of the users press "stop" in the app, on average after 50 tokens.
  They stop the answers that ramble.
- The answers that ramble would continue to `max_tokens = 4096`. The
  other answers end naturally after about 400 tokens.
- The metrics for one hour: 12.0 million tokens generated, 2.35 million
  tokens delivered to the clients.
- The stream handler catches no exception when the client goes away.

### Solution

- **Root cause.** When the client disconnects, the server does not abort
  the request. The engine generates the rest of the answer, up to 4,096
  tokens, for nobody, and it holds the KV blocks for the whole time.
- **The number.** For each request, the engine generates on average
  0.7 x 400 + 0.3 x 4,096 = 1,509 tokens, and the clients receive
  0.7 x 400 + 0.3 x 50 = 295 tokens. 295 / 1,509 = 20%. The metrics say
  2.35 / 12.0 = 20%. Four fifths of the work goes to no one.
- **The fix.** Detect the disconnect (`await request.is_disconnected()`,
  or catch `asyncio.CancelledError` in the stream), and call
  `engine.abort(request_id)`. The blocks come back at once (stage 15).
- **The guard.** Put the ratio of delivered to generated tokens on the
  dashboard. A test: disconnect a client, and check that the blocks are
  free within one step.

**The noise.** "We need more memory". The memory is full of answers that
nobody reads.

In [ ]:
generated = 0.7 * 400 + 0.3 * 4096
delivered = 0.7 * 400 + 0.3 * 50
print(f'generated {generated:.0f}, delivered {delivered:.0f}, ratio {delivered / generated:.0%}')
print(f'metrics: {2.35 / 12.0:.0%}')

**What you could ask the system**

The on-call colleague answers these questions. Each answer is a piece of evidence that the ticket does not show.

- *What happens on the server when a user presses stop?*
  The HTTP connection closes. The request continues in the engine.
- *How long does a stopped request stay in the engine?*
  Until it reaches 4,096 tokens.
- *How much of the KV memory do the stopped requests hold?*
  At 02:00, 71% of the used blocks belong to requests whose client is gone.

# Ticket 3: the dashboard says 80 ms, the users say 3 seconds

**Severity:** medium. **Reported by:** the product team.

> The TTFT on the dashboard is 80 ms at the p50. Users say that they wait
> about 3 seconds for the first word.

**Evidence**

- The code that starts the TTFT clock:

  ```python
  def admit(self, seq):
      seq.t_start = time.monotonic()
      self.running.append(seq)
  ```

- The queue wait metric, p50: 2.9 s.
- The users are on mobile networks. The team thinks that the network is
  slow.

### Solution

- **Root cause.** The TTFT clock starts at admission, not at arrival. The
  wait in the queue is not in the metric.
- **The number.** 2.9 s of queue + 0.08 s of TTFT = 2.98 s, which is what
  the users report. A client in the same data center, with no mobile
  network, also sees 3.0 s.
- **The fix.** Start the clock when the request arrives. Keep the queue
  wait as its own metric, because it has its own fix (capacity).
- **The guard.** A synthetic client that measures the TTFT from the
  outside. Alert when it differs from the server metric by more than the
  network time.

**The noise.** The mobile network. It adds about 90 ms, not 2.9 s.

In [ ]:
print('queue + dashboard TTFT =', 2.9 + 0.08, 's')

**What you could ask the system**

The on-call colleague answers these questions. Each answer is a piece of evidence that the ticket does not show.

- *What is the network round trip from the users?*
  About 90 ms at the p50.
- *What does a synthetic client in the same data center see?*
  A time to the first token of 3.0 s.
- *Where is the queue wait measured?*
  From the arrival in the HTTP handler to `admit()`.

# Ticket 4: a p99 of 480 ms that nobody sees

**Severity:** medium. **Reported by:** the SRE team.

> The fleet p99 latency on the dashboard is 480 ms. The slowest 1% of
> the requests in the client logs take more than 2 seconds.

**Evidence**

- The fleet has 10 pods. Each pod reports its own p99.
- The dashboard computes the fleet p99 as the average of the 10 values.
- 9 pods report a p99 of 200 ms. One pod reports 3,000 ms. That pod has a
  bad fan, and its GPU throttles.
- Each pod serves 10% of the traffic.

### Solution

- **Root cause.** An average of percentiles is not a percentile. The
  slowest 1% of the fleet comes almost entirely from the bad pod.
- **The number.** The dashboard: (9 x 200 + 3,000) / 10 = 480 ms. The
  real p99: the slowest 1% of all requests. The bad pod has 10% of the
  requests, so the slowest 1% of the fleet is the slowest 10% of the bad
  pod. That is its p90: 2,400 ms, five times the dashboard.
- **The fix.** Add the histogram counts of all the pods, and compute the
  percentile from the sum (`histogram_quantile` over
  `sum by (le)` in Prometheus). Replace the bad fan.
- **The guard.** Never average percentiles. Alert on the p99 of each pod
  as well, so that one bad pod is visible.

**The noise.** None. The bad fan is the real cause of the slow requests.
The average hid it.

In [ ]:
print('average of the p99s:', (9 * 200 + 3000) / 10, 'ms')
print(f'the fleet p99 is the p{100 - 1 / 0.10:.0f} of the bad pod')

**What you could ask the system**

The on-call colleague answers these questions. Each answer is a piece of evidence that the ticket does not show.

- *What is the p90 of the bad pod?*
  2,400 ms.
- *How many requests of the healthy pods take more than 1 s?*
  Almost none, fewer than 0.01%.
- *Do the pods export histograms?*
  Yes. Each pod exports the counts of a latency histogram.

# Ticket 5: streaming that arrives all at once

**Severity:** high. **Reported by:** the front-end team.

> In production, the text does not stream. The whole answer appears at
> the end. On a laptop against the pod, it streams well.

**Evidence**

- `curl` to the pod directly: the first chunk after 150 ms, then a chunk
  every 25 ms.
- `curl` through the public URL: all chunks arrive at the same moment,
  after 9.8 s.
- A long answer takes 9.8 s to generate.
- The public URL goes through an nginx ingress. Its config has no setting
  about buffering.
- The last release changed the JSON format of each chunk.

### Solution

- **Root cause.** The ingress buffers the response. nginx collects the
  chunks and sends them when its buffer fills or the response ends. An
  answer of a few KB never fills the buffer, so it arrives at the end.
- **The number.** Through the ingress, the TTFT equals the total time:
  9.8 s = 9.8 s. A stream has a TTFT much smaller than its total time.
  Directly to the pod, it does: 150 ms against 9.8 s.
- **The fix.** Turn off the buffering for the streaming routes:
  `proxy_buffering off`, or send the header `X-Accel-Buffering: no` from
  the server.
- **The guard.** A synthetic probe through the **public** path that
  fails when the TTFT is more than half of the total time.

**The noise.** The new JSON format. It changed the content of the
chunks, not when they arrive.

**What you could ask the system**

The on-call colleague answers these questions. Each answer is a piece of evidence that the ticket does not show.

- *What is the TTFT and the total time through the public URL?*
  Both are 9.8 s.
- *What does nginx do by default with a proxied response?*
  `proxy_buffering` is `on` by default.
- *Did the old release stream through the public URL?*
  The public URL is new. The old release was only reached from the internal network.

# Ticket 6: the release that made us six times faster

**Severity:** low. It goes into a press release. **Reported by:** the
marketing team.

> Release 2.3 raised the throughput from 1,900 to 11,400 tokens/s. We
> want to announce a 6x speedup.

**Evidence**

- The release notes contain: "unify the token counting in the metrics".
- The request rate and the GPU step times did not change.
- The average request has 1,500 prompt tokens and 300 output tokens.

### Solution: the engine did not change, the metric did

- **Root cause.** The new metric counts the prompt tokens too. The engine
  does the same work at the same speed.
- **The number.** (1,500 + 300) / 300 = 6.0, and 1,900 x 6.0 = 11,400.
  The "speedup" is the ratio of the two ways to count.
- **The fix.** Do not announce. Report the prompt throughput and the
  output throughput as two metrics, as vLLM does.
- **The guard.** A benchmark that runs the same fixed workload on each
  release, and compares the time, not a rate that the code computes.

In [ ]:
print('ratio of the two ways to count:', (1500 + 300) / 300)
print('1,900 x 6 =', 1900 * 6)

**What you could ask the system**

The on-call colleague answers these questions. Each answer is a piece of evidence that the ticket does not show.

- *What did the throughput metric count before the release?*
  Output tokens only.
- *What does it count now?*
  Prompt tokens and output tokens.
- *Did the time for each request change?*
  No. The p50 end-to-end latency is the same, within 1%.

# Ticket 7: production gets a third of the load test

**Severity:** medium. **Reported by:** the performance team.

> The load test with 50 users gives 1,000 tokens/s. Production has 50
> users and gives 310 tokens/s. Something in production is slow.

**Evidence**

- In the load test, each virtual user sends the next request as soon as
  the answer ends.
- In production, a user reads the answer and thinks for about 20 s
  before the next request.
- A request takes about 10 s from arrival to the last token, in both
  cases.
- The GPU utilization in production is about 35%.

### Solution: nothing is broken

- **Root cause.** The load test is a closed loop with no think time: 50
  users keep 50 requests in the system. The production users spend two
  thirds of their time reading. So production **offers** less work. The
  server is not slower. It has less to do.
- **The number.** Each user is active 10 s out of every 30 s. So
  50 x 10 / 30 = 16.7 requests are active on average, one third of 50.
  One third of 1,000 tokens/s is 333, close to the 310 measured.
- **The fix.** Nothing to fix. To measure capacity, use an open-loop
  test: send requests at a fixed rate (a Poisson trace, as in stage 16),
  and raise the rate until the promise breaks.
- **The guard.** Report the offered load (requests each second, or
  active requests) next to every throughput number.

**The noise.** The GPU utilization of 35%. It looks like a problem. It is
the same one third.

In [ ]:
active = 50 * 10 / 30
print(f'active requests: {active:.1f}, expected throughput {1000 * active / 50:.0f} tok/s')

**What you could ask the system**

The on-call colleague answers these questions. Each answer is a piece of evidence that the ticket does not show.

- *How many requests are active at a time in production, on average?*
  About 17.
- *What is the step time in production?*
  The same as in the load test at the same batch size.

### The pattern in the tickets

| The shape of the number | What it usually means | Tickets |
|---|---|---|
| All the streams stall at the same moment, and the GPU steps are flat | A blocked event loop | 1 |
| Delivered / generated far below 1 | Work for clients that left | 2 |
| The user number = the metric + another metric | A clock that starts too late | 3 |
| A percentile that is an average | The aggregation hides the tail | 4 |
| TTFT = total time | Something buffers the stream | 5 |
| A ratio that equals a ratio of two definitions | The metric changed, not the system | 6 |
| A ratio that equals a ratio of active times | Less offered load, not a slower server | 7 |

Four of these seven tickets have a correct engine. The skill in this Part is to
distrust the number before you distrust the engine.